# Wavefield Storage and Sampling Verification

The propagated fields remain float32, while saved gradient wavefields may use
float16 or bfloat16 and may be temporally subsampled. This notebook quantifies
the resulting gradient approximation and verifies CUDA asynchronous offload
when a CUDA backend is available.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, cwd / "tests"]
candidates.extend(parent / "tests" for parent in cwd.parents)
NOTEBOOK_DIR = next(
    (path for path in candidates if (path / "verification_utils.py").is_file()),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError("verification_utils.py was not found from the current directory.")
notebook_path = str(NOTEBOOK_DIR)
if notebook_path not in sys.path:
    sys.path.insert(0, notebook_path)

import verification_utils as vu

REPO_ROOT = vu.configure_local_import()
for module_name in tuple(sys.modules):
    if module_name == "DeepGPR" or module_name.startswith("DeepGPR."):
        del sys.modules[module_name]
import DeepGPR

LOADED_PACKAGE = vu.assert_local_deepgpr(DeepGPR, REPO_ROOT)
print(f"Repository root: {REPO_ROOT}")
print(f"DeepGPR package: {LOADED_PACKAGE}")


In [ ]:
import torch

torch.manual_seed(2026)
CPU = torch.device("cpu")
CHECKS = []
METADATA = vu.runtime_metadata(DeepGPR, CPU)
nx, ny, nt = 24, 30, 240
dx, dt, pml = 0.02, 3.0e-11, 4
x = torch.arange(nx, dtype=torch.float32)[:, None]
y = torch.arange(ny, dtype=torch.float32)[None, :]
anomaly = torch.exp(
    -0.5 * (((x - 15.0) / 3.5) ** 2 + ((y - 17.0) / 4.5) ** 2)
)
er0 = torch.full((nx, ny), 4.0)
se0 = torch.full((nx, ny), 3.0e-4)
er_true = er0 + 0.35 * anomaly
se_true = se0 + 1.5e-4 * anomaly
source_cpu = DeepGPR.wavelet.ricker(2.5e8, nt, dt, 4.0e-9).reshape(1, nt, 1)
source_location_cpu = torch.tensor([[[6, 10, 0]]], dtype=torch.int32)
receiver_location_cpu = torch.tensor(
    [[[6, 14, 0], [6, 18, 0], [6, 22, 0]]], dtype=torch.int32
)


In [ ]:
def forward_only(device, er_value, se_value):
    return DeepGPR.compute(
        device=device,
        dx=dx,
        dt=dt,
        source_amplitudes=source_cpu.to(device),
        source_location=source_location_cpu.to(device),
        receiver_location=receiver_location_cpu.to(device),
        er=er_value.to(device),
        se=se_value.to(device),
        pmlthick=pml,
        fdtd_order=2,
        mode=2,
    )[-1]

with torch.no_grad():
    observed_cpu = forward_only(CPU, er_true, se_true)
data_scale_cpu = observed_cpu.abs().max().clamp_min(1.0e-12)

def gradient_run(device, storage_dtype, sampling_interval, use_async_offload=False):
    er = er0.to(device).clone().requires_grad_(True)
    se = se0.to(device).clone().requires_grad_(True)
    result = DeepGPR.compute(
        device=device,
        dx=dx,
        dt=dt,
        source_amplitudes=source_cpu.to(device),
        source_location=source_location_cpu.to(device),
        receiver_location=receiver_location_cpu.to(device),
        er=er,
        se=se,
        pmlthick=pml,
        fdtd_order=2,
        mode=2,
        model_gradient_sampling_interval=sampling_interval,
        wavefield_storage_dtype=storage_dtype,
        use_async_offload=use_async_offload,
    )
    observed = observed_cpu.to(device)
    scale = data_scale_cpu.to(device)
    residual = (result[-1] - observed) / scale
    loss = 0.5 * residual.square().sum()
    loss.backward()
    vu.assert_finite("storage gradients", er.grad, se.grad, result[-1])
    return {
        "receiver": result[-1].detach().cpu(),
        "grad_er": er.grad.detach().cpu(),
        "grad_se": se.grad.detach().cpu(),
        "eall_dtype": result[0].dtype,
        "loss": float(loss.detach().cpu()),
    }

reference = gradient_run(CPU, torch.float32, 1)
vu.record_check(
    CHECKS,
    "reference saved wavefield dtype is float32",
    reference["eall_dtype"] == torch.float32,
    dtype=reference["eall_dtype"],
)


In [ ]:
storage_rows = []
storage_tolerances = {
    torch.float16: (1.0e-3, 0.99999),
    torch.bfloat16: (2.0e-3, 0.9999),
}
for storage_dtype, (relative_tolerance, cosine_tolerance) in storage_tolerances.items():
    candidate = gradient_run(CPU, storage_dtype, 1)
    receiver_difference = vu.max_abs_difference(
        candidate["receiver"], reference["receiver"]
    )
    er_relative = vu.relative_l2(candidate["grad_er"], reference["grad_er"])
    se_relative = vu.relative_l2(candidate["grad_se"], reference["grad_se"])
    er_cosine = vu.cosine_similarity(candidate["grad_er"], reference["grad_er"])
    se_cosine = vu.cosine_similarity(candidate["grad_se"], reference["grad_se"])
    row = {
        "dtype": str(storage_dtype),
        "receiver_max_abs_difference": receiver_difference,
        "er_relative_l2": er_relative,
        "se_relative_l2": se_relative,
        "er_cosine": er_cosine,
        "se_cosine": se_cosine,
    }
    storage_rows.append(row)
    vu.record_check(
        CHECKS,
        f"{storage_dtype} changes saved wavefields but not forward data",
        candidate["eall_dtype"] == storage_dtype and receiver_difference == 0.0,
        **row,
    )
    vu.record_check(
        CHECKS,
        f"{storage_dtype} gradient approximation remains controlled",
        max(er_relative, se_relative) < relative_tolerance
        and min(er_cosine, se_cosine) > cosine_tolerance,
        **row,
        relative_l2_tolerance=relative_tolerance,
        cosine_tolerance=cosine_tolerance,
    )


In [ ]:
sampling_rows = []
sampling_tolerances = {2: (2.0e-3, 0.9999), 4: (1.0e-2, 0.999)}
for sampling_interval, (relative_tolerance, cosine_tolerance) in sampling_tolerances.items():
    candidate = gradient_run(CPU, torch.float32, sampling_interval)
    er_relative = vu.relative_l2(candidate["grad_er"], reference["grad_er"])
    se_relative = vu.relative_l2(candidate["grad_se"], reference["grad_se"])
    er_cosine = vu.cosine_similarity(candidate["grad_er"], reference["grad_er"])
    se_cosine = vu.cosine_similarity(candidate["grad_se"], reference["grad_se"])
    row = {
        "sampling_interval": sampling_interval,
        "er_relative_l2": er_relative,
        "se_relative_l2": se_relative,
        "er_cosine": er_cosine,
        "se_cosine": se_cosine,
    }
    sampling_rows.append(row)
    vu.record_check(
        CHECKS,
        f"sampling interval {sampling_interval} gradient approximation",
        max(er_relative, se_relative) < relative_tolerance
        and min(er_cosine, se_cosine) > cosine_tolerance,
        **row,
        relative_l2_tolerance=relative_tolerance,
        cosine_tolerance=cosine_tolerance,
    )


In [ ]:
cuda_device = vu.selected_cuda_device()
async_row = None
if cuda_device is None:
    vu.record_skip(
        CHECKS,
        "CUDA asynchronous wavefield offload",
        "CUDA is not available on this machine.",
    )
else:
    cuda_metadata = vu.runtime_metadata(DeepGPR, cuda_device)
    direct = gradient_run(cuda_device, torch.float32, 1, False)
    offloaded = gradient_run(cuda_device, torch.float32, 1, True)
    async_row = {
        "device": str(cuda_device),
        "receiver_relative_l2": vu.relative_l2(
            offloaded["receiver"], direct["receiver"]
        ),
        "er_gradient_relative_l2": vu.relative_l2(
            offloaded["grad_er"], direct["grad_er"]
        ),
        "se_gradient_relative_l2": vu.relative_l2(
            offloaded["grad_se"], direct["grad_se"]
        ),
        "cuda_metadata": cuda_metadata,
    }
    vu.record_check(
        CHECKS,
        "CUDA asynchronous offload matches device-resident storage",
        max(
            async_row["receiver_relative_l2"],
            async_row["er_gradient_relative_l2"],
            async_row["se_gradient_relative_l2"],
        ) < 2.0e-5,
        **async_row,
        tolerance=2.0e-5,
    )


In [ ]:
vu.save_report(
    "05_wavefield_storage",
    CHECKS,
    METADATA,
    extra={
        "storage_rows": storage_rows,
        "sampling_rows": sampling_rows,
        "cuda_async_row": async_row,
    },
)
print(f"Completed {len(CHECKS)} checks, including optional checks.")
